<a href="https://colab.research.google.com/github/motebrian/BrianMotee/blob/main/cook_county_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests, time, os, shutil
import pandas as pd

base = "https://datacatalog.cookcountyil.gov/resource/ydue-e5u3.json"
cols = ("pin10, year, num_foreclosure_in_half_mile_past_5_years, "
        "nearest_new_construction_dist_ft")
limit = 50000
headers = {}   # or {"X-App-Token": "your_token"}

def get_page(params, tries=6):
    for t in range(tries):
        try:
            r = requests.get(base, params=params, headers=headers, timeout=(10, 300))
            if r.status_code == 200:
                return r.json()
            print("HTTP", r.status_code, r.text[:200])
        except requests.exceptions.RequestException as e:
            print("retry", t, type(e).__name__)
        time.sleep(5 * (t + 1))
    raise RuntimeError("page failed after retries")

drive_dir = "/content/drive/MyDrive/cook_county"

for yr in range(2020, 2026):
    path = f"/content/work/prox_{yr}.csv"
    if os.path.exists(path) or os.path.exists(f"{drive_dir}/prox_{yr}.csv"):
        continue
    frames, offset = [], 0
    while True:
        rows = get_page({"$select": cols, "$where": f"year = {yr}",
                         "$order": "pin10", "$limit": limit, "$offset": offset})
        if not rows:
            break
        frames.append(pd.DataFrame(rows))
        offset += len(rows)                      # advance by rows received
        print(yr, f"{offset:,}", end="\r")
    pd.concat(frames, ignore_index=True).to_csv(path, index=False)
    shutil.copy(path, f"{drive_dir}/prox_{yr}.csv")    # keep a copy in Drive
    print(yr, "saved", f"{offset:,}")

2020 saved 1,415,729
2021 saved 1,416,038
2022 saved 1,416,420
2023 saved 1,417,183
2024 saved 1,417,687
2025 saved 1,417,544


In [5]:
import os
from google.colab import drive

drive.mount("/content/drive")                      # skip if Drive is already mounted
os.makedirs("/content/work", exist_ok=True)        # creates the folder that was missing

!cp /content/drive/MyDrive/cook_county/av_*.csv /content/work/
!ls -lh /content/work

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 415M
-rw------- 1 root root 60M Oct 13 08:24 av_2020.csv
-rw------- 1 root root 71M Oct 13 08:24 av_2021.csv
-rw------- 1 root root 71M Oct 13 08:24 av_2022.csv
-rw------- 1 root root 71M Oct 13 08:24 av_2023.csv
-rw------- 1 root root 71M Oct 13 08:24 av_2024.csv
-rw------- 1 root root 74M Oct 13 08:24 av_2025.csv


In [8]:
import numpy as np, pandas as pd, gc

src = "/content/drive/MyDrive/cook_county"

wanted = {"pin", "year", "class", "township_code", "board_tot", "nbhd"}
LOA = {"1": .10, "2": .10, "3": .10, "4": .20, "5": .25, "9": .10}
DROP = {"E", "R", "O", "6", "7", "8"}
parts = []

for y in range(2020, 2026):
    d = pd.read_csv(f"{src}/av_{y}.csv", usecols=lambda c: c in wanted,
                    dtype="string[pyarrow]")
    d["board_tot"] = pd.to_numeric(d["board_tot"], errors="coerce").astype("float32")
    major = d["class"].str[0]
    keep = ~major.isin(DROP) & (d["board_tot"] > 0) & d["township_code"].notna()
    d, major = d[keep].copy(), major[keep]

    d["pin"] = d["pin"].str.zfill(14).astype("int64")
    d["pin10"] = d["pin"] // 10_000
    d["year"] = np.int16(y)
    d["major_class"] = major
    d["loa"] = major.map(LOA).astype("float32")
    d["market_value"] = d["board_tot"] / d["loa"]
    parts.append(d)
    print(y, f"{len(d):,}")

av = pd.concat(parts, ignore_index=True)
del parts; gc.collect()
for c in ["class", "township_code", "major_class", "nbhd"]:
    if c in av:
        av[c] = av[c].astype("category")

print(round(av.memory_usage(deep=True).sum() / 1e9, 2), "GB")
av.to_parquet(f"{src}/av_clean_2020_2025.parquet", index=False)

2020 1,764,816
2021 1,764,119
2022 1,761,900
2023 1,761,933
2024 1,761,070
2025 1,760,582
0.37 GB


In [9]:
av.head()

,pin,year,class,township_code,board_tot,pin10,major_class,loa,market_value,nbhd
0,1011000020000,2020,592,10,66817.0,101100002,5,0.25,267268.0,<NA>
1,1011000030000,2020,592,10,122000.0,101100003,5,0.25,488000.0,<NA>
2,1011000040000,2020,212,10,33020.0,101100004,2,0.10,330200.0,<NA>
3,1011000050000,2020,205,10,27820.0,101100005,2,0.10,278200.0,<NA>
4,1011000060000,2020,212,10,28869.0,101100006,2,0.10,288690.0,<NA>


In [11]:
av.drop(columns="nbhd",inplace=True)
av.head()

,pin,year,class,township_code,board_tot,pin10,major_class,loa,market_value
0,1011000020000,2020,592,10,66817.0,101100002,5,0.25,267268.0
1,1011000030000,2020,592,10,122000.0,101100003,5,0.25,488000.0
2,1011000040000,2020,212,10,33020.0,101100004,2,0.10,330200.0
3,1011000050000,2020,205,10,27820.0,101100005,2,0.10,278200.0
4,1011000060000,2020,212,10,28869.0,101100006,2,0.10,288690.0


In [12]:
#Checking if everything is OK.
expected = {2020: 1415729, 2021: 1416038, 2022: 1416420,
            2023: 1417183, 2024: 1417687, 2025: 1417544}
for y in range(2020, 2026):
    n = sum(1 for _ in open(f"{src}/prox_{y}.csv", encoding="utf-8")) - 1
    print(y, f"{n:,}", "OK" if n == expected[y] else f"MISMATCH (expected {expected[y]:,})")

2020 1,415,729 OK
2021 1,416,038 OK
2022 1,416,420 OK
2023 1,417,183 OK
2024 1,417,687 OK
2025 1,417,544 OK
